# Phase 4: Test Retrieval & Synthesis

Test the query engine:
1. **Retrieval** - Find relevant chunks with metadata filtering
2. **Synthesis** - Generate answers with citations
3. **Recommendations** - Suggest episodes based on context


In [1]:
import sys
sys.path.insert(0, '../src')

from retriever import retrieve_chunks, retrieve_by_guest, get_all_guests, get_episodes_by_topic
from synthesizer import synthesize_answer, recommend_episodes, quick_answer


/Users/nanditakrishnan/llm/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Basic semantic search
results = retrieve_chunks("How to run effective 1:1 meetings", n_results=5)

print(f"Found {len(results)} results\n")
for r in results:
    print(f"📌 {r.guest_name} ({r.guest_role})")
    print(f"   ⏱️  {r.start_timestamp} - {r.end_timestamp}")
    print(f"   📊 Tactical: {r.tactical_score}/10 | Distance: {r.distance:.3f}")
    print(f"   💬 {r.text[:200]}...")
    print()


Found 5 results

📌 Evan LaPointe (founder / PM)
   ⏱️  00:23:16 - 00:23:32
   📊 Tactical: 8/10 | Distance: 425.149
   💬 Lenny Rachitsky: Are there things that you've found people can change in the way they work based on the way the brain operates, whether it's run better meetings, be better influence? What are some thi...

📌 Nikhyl Singhal (exec)
   ⏱️  01:20:11 - 01:21:29
   📊 Tactical: 7/10 | Distance: 426.002
   💬 Nikhyl Singhal: So what's interesting is every quarter, in my current teams, even in my past teams, I talk about our meetings like a product. We're on version seven in my team, and so we're like, "Hey...

📌 Naomi Gleit (PM)
   ⏱️  01:07:48 - 01:09:11
   📊 Tactical: 8/10 | Distance: 426.258
   💬 Naomi Gleit: If somebody joins the meeting, say, five minutes late, they should know exactly where in the agenda you are in the meeting and what is being discussed based on catching up from the visual...

📌 Chandra Janakiraman (founder/PM/designer)
   ⏱️  00:58:28 - 00:59:47
   📊 Ta

In [3]:
# Filtered search - only tactical episodes (score >= 7)
results = retrieve_chunks(
    "frameworks for prioritization", 
    n_results=5,
    min_tactical_score=7
)

print("Highly tactical episodes on prioritization:\n")
for r in results:
    print(f"📌 {r.guest_name} (Tactical: {r.tactical_score}/10)")
    print(f"   Topics: {', '.join(r.topics[:3]) if r.topics else 'N/A'}")
    print(f"   💬 {r.text[:200]}...")
    print()


Highly tactical episodes on prioritization:

📌 Maggie Crowley (Tactical: 8/10)
   Topics: Product Management, Prioritization, Simplifying Complexity in Product Development
   💬 Maggie Crowley: Prioritization is a tough word because there's so much wrapped up in that and what it means to prioritize. And I've worked for people who wanted to understand the formula for prioritiz...

📌 Manik Gupta (Tactical: 7/10)
   Topics: building successful consumer products, structuring and hiring product teams, consumer stack
   💬 Manik Gupta: Number two is strong focus and prioritization. You can apply strong focus and prioritization to anything in life, but I think it's even more important for consumer products. Because often...

📌 Sachin Monga (Tactical: 8/10)
   Topics: Substack, product-market fit, growth
   💬 Sachin Monga: I think going back to the previous point, a lot of people really thrive in that kind of environment, where if we do this thing really well, it is going to directly trade-off a

In [4]:
# Quick answer - fast, concise response (uses smaller 3B model)
query = "What's the best way to say no to feature requests?"

print(f"Query: {query}\n")
print("Answer:")
print(quick_answer(query))


Query: What's the best way to say no to feature requests?

Answer:
To say no to feature requests effectively, it's essential to establish clear boundaries and prioritize your goals. Having pre-written templates or auto-suggestions can help you communicate your decision in a polite and considerate manner. This approach can help you avoid overcommitting and maintain a healthy work-life balance.


In [8]:
# Full synthesis - comprehensive answer with citations
query = "How should I think about pricing my SaaS product?"

result = synthesize_answer(query, n_chunks=8)

print(f"Query: {result.query}\n")
print("=" * 60)
print("SYNTHESIZED ANSWER")
print("=" * 60)
print(result.answer)

print("\n" + "=" * 60)
print(f"SOURCES ({len(result.sources)} chunks)")
print("=" * 60)
for s in result.sources:
    print(f"  • {s['guest']} ({s['role']}) @ {s['timestamp']}")


Query: How should I think about pricing my SaaS product?

SYNTHESIZED ANSWER
Pricing your SaaS product is a complex and nuanced process that requires careful consideration of various factors, including your target market, competition, product features, and customer needs. Based on the provided podcast transcript excerpts, here are some key takeaways and recommendations for pricing your SaaS product:

1. **Choose the right pricing model**: Madhavan Ramanujam suggests that B2B SaaS companies can use subscription, pay-as-you-go, freemium, or tiered pricing models. Consider your product's value proposition, customer needs, and competition when selecting a pricing model.
2. **Understand your pricing metrics**: Ramanujam emphasizes the importance of clear metrics for tracking and attributing value to your customers. Consider using seat-based, flat-based, or usage-based pricing metrics, depending on your product and business model.
3. **Consider hybrid pricing models**: HubSpot's hybrid model

In [6]:
# Episode recommendations based on your context
query = "I'm a new PM at a Series A startup trying to build a roadmap"

result = recommend_episodes(query, n_chunks=15)

print(f"Request: {result.query}\n")
print("=" * 60)
print("RECOMMENDED EPISODES")
print("=" * 60)
print(result.answer)


Request: I'm a new PM at a Series A startup trying to build a roadmap

RECOMMENDED EPISODES
Based on your context as a new PM at a Series A startup trying to build a roadmap, here are the top episode recommendations, ordered from most to least relevant:

1. **Ian McAllister (exec)**: Ian McAllister discusses his experience as a product manager and leader, sharing insights on what separates top performers from others and how to implement the working backwards process. This episode is highly relevant as it provides practical advice on how to build a roadmap, prioritize projects, and stay focused on key objectives.

2. **Chandra Janakiraman (founder/PM/designer)**: Chandra Janakiraman shares his operator's guide to strategy, a 5-step process for developing great strategies and products. As a new PM, you'll benefit from learning how to develop effective strategies and products that drive business growth.

3. **Mayur Kamat (founder / PM)**: Mayur Kamat shares his experiences as a product ma

In [7]:
# Browse episodes by topic
topic = "AI"
episodes = get_episodes_by_topic(topic)

print(f"Episodes covering '{topic}': {len(episodes)}\n")
for guest, meta in episodes[:8]:
    print(f"• {guest} ({meta.guest_role})")
    print(f"  {meta.one_line_summary}")
    print()


Episodes covering 'AI': 70



AttributeError: 'dict' object has no attribute 'guest_role'

In [ ]:
# Try your own query!
my_query = "How do top PMs handle disagreements with engineers?"

result = synthesize_answer(my_query)
print(result.answer)
